# Upload the 16 CC-Perturb-seq analysis files to IGVF

This notebook performs the complete parent-first production workflow for the
four D0-D3 AnalysisSets:

1. Use explicitly hard-coded first-pass software versions for the two analysis
   repositories. No Git clone, fetch, SSH authentication, or GitHub API access
   is performed.
2. Generate Software, SoftwareVersion, AnalysisStep, and AnalysisStepVersion
   metadata from those fixed values.
3. Query the production Portal before every parent-object POST, and reuse
   existing shared Software/SoftwareVersion records by their Portal unique keys.
4. Dry-run and, only after an explicit confirmation token, create missing
   parent objects in dependency order.
5. Revalidate the 16 final local files and their MD5 checksums.
6. Dry-run and upload the four h5ad MatrixFiles.
7. Verify those h5ad records, then dry-run and upload the four h5mu and eight
   tabular files.
8. Reconcile all 16 records and export their accessions and Portal links.

## First-pass provenance limitation

Both repositories are assigned the hard-coded semantic version `v0.1.0`.
The repository URLs are recorded, but this notebook does **not** claim an exact
Git commit or tag. This is intended to unblock an initial IGVF submission.
Before final public release, reconcile the SoftwareVersion metadata with the
exact tag or commit used if DACC requires that level of provenance.

No production write occurs until the corresponding confirmation token is set
in the configuration cell and that cell is rerun.

## Notebook environment

The notebook kernel does not need to be the `igvf_utils` conda environment. All Portal dry-run and submission commands invoke the explicit external interpreter `/oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/bin/python`. Python 3.9 deprecation warnings from that subprocess are non-blocking.


In [1]:
from pathlib import Path
import re

# Final files and metadata generated by 01_make_igvf_upload_ready_files.ipynb.
OUTPUT_ROOT = Path(
    "/oak/stanford/groups/engreitz/Projects/IGVF/"
    "CC-Perturb-seq/300gene/analysis_submission"
)
METADATA_DIR = OUTPUT_ROOT / "metadata"
LOG_DIR = OUTPUT_ROOT / "logs" / "igvf_utils_analysis_upload"
UPLOAD_WORK_DIR = OUTPUT_ROOT / "_upload_work"
ONE_ROW_DIR = UPLOAD_WORK_DIR / "one_row_tsv"
PARENT_METADATA_DIR = UPLOAD_WORK_DIR / "parent_metadata"
RESOLVED_METADATA_DIR = UPLOAD_WORK_DIR / "resolved_file_metadata"

READY_MARKER = OUTPUT_ROOT / "READY_TO_UPLOAD_16_FILES.txt"

# Source metadata written by notebook 01. These are never modified in place.
SOURCE_MASTER_MANIFEST = METADATA_DIR / "upload_manifest_16_files.tsv"
SOURCE_H5AD_TSV = METADATA_DIR / "01_matrix_file_h5ad.tsv"
SOURCE_H5MU_TSV = METADATA_DIR / "02_matrix_file_h5mu.tsv"
SOURCE_TABULAR_TSV = METADATA_DIR / "03_tabular_file_step1.tsv"

# Copies used by this notebook after inserting the fixed AnalysisStepVersion aliases.
MASTER_MANIFEST = RESOLVED_METADATA_DIR / "upload_manifest_16_files_resolved_versions.tsv"
H5AD_TSV = RESOLVED_METADATA_DIR / "01_matrix_file_h5ad_resolved_versions.tsv"
H5MU_TSV = RESOLVED_METADATA_DIR / "02_matrix_file_h5mu_resolved_versions.tsv"
TABULAR_TSV = RESOLVED_METADATA_DIR / "03_tabular_file_step1_resolved_versions.tsv"

IGVF_PYTHON = Path(
    "/oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/bin/python"
)
IU_REGISTER = Path(
    "/oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/igvf_utils/"
    "igvf_utils/MetaDataRegistration/iu_register.py"
)

IGVF_MODE = "prod"
IGVF_API_BASE = "https://api.data.igvf.org"
IGVF_PORTAL_BASE = "https://data.igvf.org"
AWARD = "/awards/HG011972/"
LAB = "/labs/jesse-engreitz/"

ANALYSIS_SETS = {
    "D0": "IGVFDS0463XTUH",
    "D1": "IGVFDS0991DBQZ",
    "D2": "IGVFDS4402ZGWJ",
    "D3": "IGVFDS2307REVJ",
}
REFERENCE_FILE = "IGVFFI7217ZMJZ"

# -------------------------------------------------------------------------
# Hard-coded first-pass software provenance.
#
# This deliberately avoids all Git/GitHub access. The semantic versions and
# repository URLs are sufficient for a first-pass Portal submission, but they
# do not identify an exact commit. Reconcile them before final release if DACC
# requests immutable commit-level provenance.
# -------------------------------------------------------------------------
HARD_CODED_SOFTWARE_VERSIONS = {
    "preprocessing": {
        "title": "CC-Perturb-seq 300-gene preprocessing",
        "software_name": "cc-perturb-seq-300genes-preprocessing",
        "software_alias_base": "CC-Perturb-seq",
        "portal_version": "v0.1.0",
        "software_source_url": (
            "https://github.com/EngreitzLab/"
            "CC-Perturb-seq_300genes_preprocessing"
        ),
        "software_version_source_url": (
            "https://github.com/EngreitzLab/"
            "CC-Perturb-seq_300genes_preprocessing"
        ),
        "version_source": "hardcoded_first_pass",
        "provenance_note": (
            "Hard-coded semantic version v0.1.0 for first-pass IGVF "
            "submission; exact tag/commit not recorded in this notebook."
        ),
    },
    "perturbnmf": {
        "title": "PerturbNMF",
        "software_name": "perturbnmf",
        "software_alias_base": "PerturbNMF",
        "portal_version": "v0.1.0",
        "software_source_url": (
            "https://github.com/EngreitzLab/PerturbNMF"
        ),
        "software_version_source_url": (
            "https://github.com/EngreitzLab/PerturbNMF"
        ),
        "version_source": "hardcoded_first_pass",
        "provenance_note": (
            "Hard-coded semantic version v0.1.0 for first-pass IGVF "
            "submission; exact tag/commit not recorded in this notebook."
        ),
    },
}

PORTAL_VERSION_PATTERN = re.compile(
    r"^v(?!0\.0\.0$)([0-9]+[A-Za-z0-9]*(\.[0-9]+[A-Za-z0-9]*){0,3})$"
)

# Definitive but potentially time-consuming. Leave True for the first upload.
VERIFY_MD5_BEFORE_UPLOAD = True

# Real parent-object creation and file upload remain blocked until these values
# are edited exactly and the configuration cell is rerun.
PARENT_REAL_CONFIRMATION = ""
H5AD_REAL_CONFIRMATION = ""
DEPENDENT_REAL_CONFIRMATION = ""

EXPECTED_PARENT_TOKEN = "CREATE_MISSING_ANALYSIS_PARENT_OBJECTS_IN_IGVF_PROD"
EXPECTED_H5AD_TOKEN = "UPLOAD_4_H5AD_FILES_TO_IGVF_PROD"
EXPECTED_DEPENDENT_TOKEN = "UPLOAD_12_DEPENDENT_FILES_TO_IGVF_PROD"

# Small pause between production POSTs reduces API bursts.
SECONDS_BETWEEN_REAL_UPLOADS = 2.0

for directory in (
    LOG_DIR,
    UPLOAD_WORK_DIR,
    ONE_ROW_DIR,
    PARENT_METADATA_DIR,
    RESOLVED_METADATA_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("READY_MARKER:", READY_MARKER)
print("IGVF_PYTHON:", IGVF_PYTHON)
print("IU_REGISTER:", IU_REGISTER)
print("Software version mode: HARD-CODED FIRST PASS; no Git access")
for key, spec in HARD_CODED_SOFTWARE_VERSIONS.items():
    print(f"  {key}: {spec['portal_version']} | {spec['software_source_url']}")
print("VERIFY_MD5_BEFORE_UPLOAD:", VERIFY_MD5_BEFORE_UPLOAD)
print("Parent creation authorized:", PARENT_REAL_CONFIRMATION == EXPECTED_PARENT_TOKEN)
print("H5AD real upload authorized:", H5AD_REAL_CONFIRMATION == EXPECTED_H5AD_TOKEN)
print("Dependent real upload authorized:", DEPENDENT_REAL_CONFIRMATION == EXPECTED_DEPENDENT_TOKEN)


OUTPUT_ROOT: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission
READY_MARKER: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/READY_TO_UPLOAD_16_FILES.txt
IGVF_PYTHON: /oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/bin/python
IU_REGISTER: /oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/igvf_utils/igvf_utils/MetaDataRegistration/iu_register.py
Software version mode: HARD-CODED FIRST PASS; no Git access
  preprocessing: v0.1.0 | https://github.com/EngreitzLab/CC-Perturb-seq_300genes_preprocessing
  perturbnmf: v0.1.0 | https://github.com/EngreitzLab/PerturbNMF
VERIFY_MD5_BEFORE_UPLOAD: True
Parent creation authorized: False
H5AD real upload authorized: False
Dependent real upload authorized: False


## Helper functions

Run this cell before all other code cells.

In [2]:
import base64
import csv
import getpass
import hashlib
import json
import os
import re
import subprocess
import time
from datetime import datetime, timezone
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.parse import quote, urlencode
from urllib.request import Request, urlopen

import pandas as pd

ERROR_PATTERNS = re.compile(
    r"Traceback|ValidationFailure|Failed to POST|Unprocessable|Forbidden|"
    r"Unauthorized|Permission denied|401 Client Error|403 Client Error|"
    r"409 Client Error|KeyError|JSONDecodeError|ReadTimeout|ConnectionError",
    flags=re.IGNORECASE,
)
ACCESSION_PATTERN = re.compile(r"IGVFFI[A-Z0-9]+")


def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def is_blank(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, (str, bytes)):
        return not str(value).strip()
    try:
        return bool(pd.isna(value))
    except (TypeError, ValueError):
        return False


def normalize_identifier(value: Any) -> str:
    if isinstance(value, dict):
        value = value.get("@id") or value.get("accession") or ""
    if is_blank(value):
        return ""
    text = str(value).strip()
    if not text:
        return ""
    if text.startswith("http://") or text.startswith("https://"):
        text = text.rstrip("/").split("/")[-1]
    elif text.startswith("/"):
        text = text.rstrip("/").split("/")[-1]
    return text


def normalize_aliases(value: Any) -> list[str]:
    if isinstance(value, list):
        return [str(item) for item in value]
    if isinstance(value, str) and value:
        return [value]
    return []


def split_identifiers(value: Any) -> list[str]:
    if is_blank(value):
        return []
    return [item.strip() for item in str(value).split(",") if item.strip()]


def credential_headers() -> dict[str, str]:
    key = os.environ.get("IGVF_API_KEY", "")
    secret = os.environ.get("IGVF_SECRET_KEY", "")
    if not key or not secret:
        raise RuntimeError("IGVF_API_KEY and IGVF_SECRET_KEY are not set in this kernel")
    encoded = base64.b64encode(f"{key}:{secret}".encode()).decode()
    return {
        "Authorization": f"Basic {encoded}",
        "Accept": "application/json",
        "User-Agent": "cc-perturbseq-analysis-upload-notebook/1.0",
    }


def portal_get(
    identifier: str,
    *,
    attempts: int = 5,
    wait_seconds: float = 2.0,
) -> dict[str, Any] | None:
    ident = normalize_identifier(identifier)
    if not ident:
        raise ValueError("Cannot query an empty Portal identifier")
    encoded_ident = quote(ident, safe=":@._-")
    url = (
        f"{IGVF_API_BASE}/{encoded_ident}/"
        "?format=json&datastore=database&frame=object"
    )
    last_error: Exception | None = None
    for attempt in range(1, attempts + 1):
        request = Request(url, headers=credential_headers(), method="GET")
        try:
            with urlopen(request, timeout=60) as response:
                return json.loads(response.read().decode("utf-8"))
        except HTTPError as exc:
            if exc.code == 404:
                return None
            if exc.code in {401, 403}:
                body = exc.read().decode("utf-8", errors="replace")[:500]
                raise RuntimeError(
                    f"Portal authentication/permission failure for {ident}: "
                    f"HTTP {exc.code}: {body}"
                ) from exc
            last_error = exc
            if exc.code not in {429, 500, 502, 503, 504} or attempt == attempts:
                body = exc.read().decode("utf-8", errors="replace")[:500]
                raise RuntimeError(
                    f"Portal GET failed for {ident}: HTTP {exc.code}: {body}"
                ) from exc
        except (URLError, TimeoutError, json.JSONDecodeError) as exc:
            last_error = exc
            if attempt == attempts:
                raise RuntimeError(f"Portal GET failed for {ident}: {exc}") from exc
        time.sleep(wait_seconds * attempt)
    raise RuntimeError(f"Portal GET failed for {ident}: {last_error}")



def portal_search(
    object_type: str,
    filters: dict[str, Any] | None = None,
    *,
    limit: str | int = "all",
    attempts: int = 5,
    wait_seconds: float = 2.0,
) -> list[dict[str, Any]]:
    """Run an authenticated read-only Portal search and return @graph.

    The IGVF search endpoint returns HTTP 404 when there are no matches, which
    is normalized here to an empty list.
    """

    params: list[tuple[str, str]] = [
        ("type", object_type),
        ("limit", str(limit)),
        ("format", "json"),
        ("datastore", "database"),
    ]
    for key, value in (filters or {}).items():
        if isinstance(value, (list, tuple, set)):
            params.extend((key, str(item)) for item in value)
        elif not is_blank(value):
            params.append((key, str(value)))

    url = f"{IGVF_API_BASE}/search/?{urlencode(params, doseq=True)}"
    last_error: Exception | None = None

    for attempt in range(1, attempts + 1):
        request = Request(url, headers=credential_headers(), method="GET")
        try:
            with urlopen(request, timeout=60) as response:
                payload = json.loads(response.read().decode("utf-8"))
            graph = payload.get("@graph", [])
            if not isinstance(graph, list):
                raise RuntimeError(
                    f"Unexpected Portal search response for {object_type}: "
                    "@graph is not a list"
                )
            return graph
        except HTTPError as exc:
            if exc.code == 404:
                return []
            if exc.code in {401, 403}:
                body = exc.read().decode("utf-8", errors="replace")[:500]
                raise RuntimeError(
                    f"Portal authentication/permission failure during "
                    f"{object_type} search: HTTP {exc.code}: {body}"
                ) from exc
            last_error = exc
            if exc.code not in {429, 500, 502, 503, 504} or attempt == attempts:
                body = exc.read().decode("utf-8", errors="replace")[:500]
                raise RuntimeError(
                    f"Portal search failed for {object_type}: "
                    f"HTTP {exc.code}: {body}"
                ) from exc
        except (URLError, TimeoutError, json.JSONDecodeError) as exc:
            last_error = exc
            if attempt == attempts:
                raise RuntimeError(
                    f"Portal search failed for {object_type}: {exc}"
                ) from exc
        time.sleep(wait_seconds * attempt)

    raise RuntimeError(
        f"Portal search failed for {object_type}: {last_error}"
    )


def portal_full_object(search_hit: dict[str, Any]) -> dict[str, Any]:
    """Fetch a full object for a partial search result when possible."""

    identifier = (
        search_hit.get("@id")
        or search_hit.get("accession")
        or search_hit.get("uuid")
        or search_hit.get("name")
    )
    if not identifier:
        return search_hit
    try:
        return portal_get(str(identifier)) or search_hit
    except Exception:
        return search_hit


def portal_object_identifiers(obj: dict[str, Any] | None) -> set[str]:
    """Collect normalized identifiers that can refer to a Portal object."""

    if not obj:
        return set()
    identifiers = {
        normalize_identifier(obj.get("@id")),
        normalize_identifier(obj.get("accession")),
        normalize_identifier(obj.get("uuid")),
        normalize_identifier(obj.get("name")),
    }
    identifiers.update(
        normalize_identifier(alias)
        for alias in normalize_aliases(obj.get("aliases"))
    )
    return {identifier for identifier in identifiers if identifier}


def portal_find_software_by_name(name: str) -> dict[str, Any] | None:
    """Resolve a Software object by its globally unique `name` field."""

    hits = portal_search(
        "Software",
        {"name": name},
        limit="all",
    )
    exact = []
    for hit in hits:
        obj = portal_full_object(hit)
        if str(obj.get("name") or "") == name:
            exact.append(obj)

    unique: dict[str, dict[str, Any]] = {}
    for obj in exact:
        key = (
            normalize_identifier(obj.get("uuid"))
            or normalize_identifier(obj.get("@id"))
            or normalize_identifier(obj.get("name"))
        )
        unique[key] = obj

    if not unique:
        return None
    if len(unique) > 1:
        raise RuntimeError(
            f"More than one Software record has name={name!r}: "
            f"{sorted(unique)}"
        )
    return next(iter(unique.values()))


def software_link_matches(observed: Any, expected: str) -> bool:
    """Compare a Software link with an expected Software name/identifier."""

    expected_norm = normalize_identifier(expected)
    observed_norm = normalize_identifier(observed)
    if observed_norm == expected_norm:
        return True

    observed_obj: dict[str, Any] | None
    if isinstance(observed, dict):
        observed_obj = observed
    elif observed_norm:
        try:
            observed_obj = portal_get(observed_norm)
        except Exception:
            observed_obj = None
    else:
        observed_obj = None

    if observed_obj and str(observed_obj.get("name") or "") == expected_norm:
        return True

    try:
        expected_obj = (
            portal_get(expected_norm)
            or portal_find_software_by_name(expected_norm)
        )
    except Exception:
        expected_obj = None

    if observed_obj and expected_obj:
        return bool(
            portal_object_identifiers(observed_obj)
            & portal_object_identifiers(expected_obj)
        )
    return False


def portal_find_software_version(
    version: str,
    software_identifier: str,
) -> dict[str, Any] | None:
    """Resolve SoftwareVersion by the Portal composite key software+version."""

    hits = portal_search(
        "SoftwareVersion",
        {"version": version},
        limit="all",
    )
    matches = []
    for hit in hits:
        obj = portal_full_object(hit)
        if str(obj.get("version") or "") != version:
            continue
        if software_link_matches(obj.get("software"), software_identifier):
            matches.append(obj)

    unique: dict[str, dict[str, Any]] = {}
    for obj in matches:
        key = (
            normalize_identifier(obj.get("uuid"))
            or normalize_identifier(obj.get("@id"))
            or "|".join(sorted(portal_object_identifiers(obj)))
        )
        unique[key] = obj

    if not unique:
        return None
    if len(unique) > 1:
        raise RuntimeError(
            "More than one SoftwareVersion matches "
            f"software={software_identifier!r}, version={version!r}: "
            f"{sorted(unique)}"
        )
    return next(iter(unique.values()))


def best_portal_link(obj: dict[str, Any]) -> str:
    """Return a stable link accepted by Portal linkTo fields."""

    return str(
        obj.get("@id")
        or obj.get("accession")
        or obj.get("uuid")
        or obj.get("name")
        or (normalize_aliases(obj.get("aliases")) or [""])[0]
    )


def refresh_parent_links(records: list[dict[str, Any]]) -> None:
    """Replace provisional aliases/names with actual Portal @ids when available."""

    record_map = {record["key"]: record for record in records}

    for record in records:
        profile = record["profile"]
        dependencies = [
            record_map[key]
            for key in record.get("depends_on", [])
            if key in record_map
        ]

        if profile == "software_version":
            software_dependencies = [
                dep for dep in dependencies if dep["profile"] == "software"
            ]
            if software_dependencies:
                obj = parent_record_object(software_dependencies[0])
                if obj:
                    record["row"]["software"] = best_portal_link(obj)

        elif profile == "analysis_step" and "parents" in record["row"]:
            step_dependencies = [
                dep for dep in dependencies if dep["profile"] == "analysis_step"
            ]
            links = []
            for dep in step_dependencies:
                obj = parent_record_object(dep)
                if obj:
                    links.append(best_portal_link(obj))
            if links:
                record["row"]["parents"] = ",".join(links)

        elif profile == "analysis_step_version":
            step_dependencies = [
                dep for dep in dependencies if dep["profile"] == "analysis_step"
            ]
            version_dependencies = [
                dep
                for dep in dependencies
                if dep["profile"] == "software_version"
            ]

            if step_dependencies:
                obj = parent_record_object(step_dependencies[0])
                if obj:
                    record["row"]["analysis_step"] = best_portal_link(obj)

            version_links = []
            for dep in version_dependencies:
                obj = parent_record_object(dep)
                if obj:
                    version_links.append(best_portal_link(obj))
            if version_links:
                record["row"]["software_versions"] = ",".join(version_links)

def portal_url(accession: str) -> str:
    return f"{IGVF_PORTAL_BASE}/{accession}/" if accession else ""


def record_comparison(row: dict[str, str], obj: dict[str, Any]) -> list[str]:
    issues: list[str] = []
    expected_alias = row["aliases"]
    aliases = normalize_aliases(obj.get("aliases"))
    if expected_alias not in aliases:
        issues.append(f"alias missing on Portal: {expected_alias}")

    expected_md5 = row["md5sum"]
    portal_md5 = str(obj.get("md5sum") or "")
    if portal_md5 != expected_md5:
        issues.append(f"md5sum mismatch: expected={expected_md5}, portal={portal_md5}")

    expected_file_set = normalize_identifier(row["file_set"])
    portal_file_set = normalize_identifier(obj.get("file_set"))
    if portal_file_set != expected_file_set:
        issues.append(
            f"file_set mismatch: expected={expected_file_set}, portal={portal_file_set}"
        )

    expected_format = row.get("file_format", "")
    portal_format = str(obj.get("file_format") or "")
    if portal_format and portal_format != expected_format:
        issues.append(
            f"file_format mismatch: expected={expected_format}, portal={portal_format}"
        )

    expected_content = row.get("content_type", "")
    portal_content = str(obj.get("content_type") or "")
    if portal_content and portal_content != expected_content:
        issues.append(
            f"content_type mismatch: expected={expected_content}, portal={portal_content}"
        )

    expected_step_version = row.get("analysis_step_version", "")
    portal_step_version = obj.get("analysis_step_version")
    if expected_step_version and portal_step_version and not portal_link_matches(
        portal_step_version,
        expected_step_version,
    ):
        issues.append(
            "analysis_step_version mismatch: "
            f"expected={expected_step_version}, "
            f"portal={normalize_identifier(portal_step_version)}"
        )

    upload_status = str(obj.get("upload_status") or "").lower()
    if upload_status in {"file not found", "invalidated"}:
        issues.append(f"Portal upload_status is {upload_status!r}")
    object_status = str(obj.get("status") or "").lower()
    if object_status in {"revoked", "archived", "deleted"}:
        issues.append(f"Portal object status is {object_status!r}")
    return issues


def wait_for_record(
    alias: str,
    *,
    attempts: int = 12,
    wait_seconds: float = 5.0,
) -> dict[str, Any] | None:
    for attempt in range(1, attempts + 1):
        obj = portal_get(alias, attempts=3)
        if obj:
            return obj
        if attempt < attempts:
            time.sleep(wait_seconds)
    return None


def md5sum(path: Path, block_size: int = 16 * 1024 * 1024) -> str:
    digest = hashlib.md5()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(block_size), b""):
            digest.update(block)
    return digest.hexdigest()


def load_metadata(path: Path, profile: str, stage: str) -> pd.DataFrame:
    frame = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
    if frame.empty:
        raise ValueError(f"Metadata file is empty: {path}")
    if any(column.startswith("#") for column in frame.columns):
        raise ValueError(f"Response/helper columns beginning with # found in {path}")
    empty_columns = [column for column in frame.columns if not frame[column].str.strip().any()]
    if empty_columns:
        print(f"Removing all-empty columns from {path.name}: {empty_columns}")
        frame = frame.drop(columns=empty_columns)
    frame["_profile"] = profile
    frame["_stage"] = stage
    frame["_source_tsv"] = str(path)
    return frame


def safe_slug(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text).strip("_")


def write_one_row_tsv(row: dict[str, str], profile: str, index: int) -> Path:
    alias = row["aliases"]
    path = ONE_ROW_DIR / f"{index:02d}_{profile}_{safe_slug(alias)}.tsv"
    payload = {
        key: "" if is_blank(value) else str(value)
        for key, value in row.items()
        if not key.startswith("_")
    }
    fields = [key for key, value in payload.items() if value.strip()]
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=fields,
            delimiter="\t",
            lineterminator="\n",
            extrasaction="ignore",
        )
        writer.writeheader()
        writer.writerow(payload)
    return path


def run_command_streaming(command: list[str], log_path: Path) -> int:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print("Command:", " ".join(command))
    print("Log:", log_path)
    with log_path.open("w", encoding="utf-8") as log_handle:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=os.environ.copy(),
        )
        try:
            assert process.stdout is not None
            for line in process.stdout:
                print(line, end="")
                log_handle.write(line)
                log_handle.flush()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
            raise
        return process.wait()


def run_iu_register(
    row: dict[str, str],
    *,
    dry_run: bool,
    ordinal: int,
) -> dict[str, Any]:
    profile = row["_profile"]
    alias = row["aliases"]
    row_tsv = write_one_row_tsv(row, profile, ordinal)
    mode_label = "dryrun" if dry_run else "submit"
    log_path = LOG_DIR / (
        f"{ordinal:02d}_{profile}_{safe_slug(alias)}_{mode_label}_"
        f"{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    )
    command = [
        str(IGVF_PYTHON),
        str(IU_REGISTER),
        "-m",
        IGVF_MODE,
        "-p",
        profile,
    ]
    if dry_run:
        command.append("-d")
    command.extend(["-i", str(row_tsv)])

    started = now_utc()
    return_code = run_command_streaming(command, log_path)
    finished = now_utc()
    text = log_path.read_text(encoding="utf-8", errors="replace")
    result = {
        "alias": alias,
        "profile": profile,
        "stage": row["_stage"],
        "dry_run": dry_run,
        "return_code": return_code,
        "dry_run_marker": "DRY RUN is enabled" in text,
        "error_pattern_found": bool(ERROR_PATTERNS.search(text)),
        "aws_upload_success_message": "AWS upload successful" in text,
        "accessions_in_log": ",".join(sorted(set(ACCESSION_PATTERN.findall(text)))),
        "row_tsv": str(row_tsv),
        "log_path": str(log_path),
        "started_utc": started,
        "finished_utc": finished,
    }
    return result


def local_aws_success_for_alias(alias: str) -> bool:
    pattern = f"*_{safe_slug(alias)}_submit_*.log"
    for log_path in LOG_DIR.glob(pattern):
        try:
            if "AWS upload successful" in log_path.read_text(
                encoding="utf-8", errors="replace"
            ):
                return True
        except OSError:
            continue
    return False


def existing_record_is_safely_reusable(
    row: dict[str, str], obj: dict[str, Any]
) -> tuple[bool, list[str]]:
    issues = record_comparison(row, obj)
    if issues:
        return False, issues
    upload_status = str(obj.get("upload_status") or "").lower()
    if upload_status in {"validated", "validation exempted"}:
        return True, []
    if local_aws_success_for_alias(row["aliases"]):
        return True, []
    if upload_status == "pending":
        return False, [
            "matching Portal record is still pending and no local log confirms "
            "'AWS upload successful'; review before skipping or retrying"
        ]
    return True, []


def dry_run_rows(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    results: list[dict[str, Any]] = []
    for ordinal, row in enumerate(frame.to_dict("records"), start=1):
        alias = row["aliases"]
        existing = portal_get(alias)
        if existing:
            reusable, issues = existing_record_is_safely_reusable(row, existing)
            if not reusable:
                raise RuntimeError(
                    f"{alias} already exists but cannot be safely skipped: "
                    + "; ".join(issues)
                )
            results.append({
                "alias": alias,
                "profile": row["_profile"],
                "stage": row["_stage"],
                "dry_run": True,
                "return_code": 0,
                "dry_run_marker": True,
                "error_pattern_found": False,
                "aws_upload_success_message": False,
                "accessions_in_log": str(existing.get("accession") or ""),
                "row_tsv": "",
                "log_path": "",
                "started_utc": now_utc(),
                "finished_utc": now_utc(),
                "status": "ALREADY_EXISTS_MATCH",
            })
            print(f"SKIP dry run; matching record already exists: {alias}")
            continue

        result = run_iu_register(row, dry_run=True, ordinal=ordinal)
        if (
            result["return_code"] != 0
            or not result["dry_run_marker"]
            or result["error_pattern_found"]
        ):
            result["status"] = "DRY_RUN_FAILED"
            results.append(result)
            output = pd.DataFrame(results)
            output.to_csv(
                UPLOAD_WORK_DIR / f"{label}_dryrun_results.tsv",
                sep="\t",
                index=False,
                lineterminator="\n",
            )
            raise RuntimeError(
                f"Dry run failed for {alias}. Review {result['log_path']}"
            )
        result["status"] = "DRY_RUN_PASS"
        results.append(result)

    output = pd.DataFrame(results)
    output_path = UPLOAD_WORK_DIR / f"{label}_dryrun_results.tsv"
    output.to_csv(output_path, sep="\t", index=False, lineterminator="\n")
    print(f"Dry-run results: {output_path}")
    print(output[["alias", "profile", "status", "log_path"]].to_string(index=False))
    return output


def submit_rows(frame: pd.DataFrame, label: str) -> pd.DataFrame:
    results: list[dict[str, Any]] = []
    for ordinal, row in enumerate(frame.to_dict("records"), start=1):
        alias = row["aliases"]
        existing = portal_get(alias)
        if existing:
            reusable, issues = existing_record_is_safely_reusable(row, existing)
            if not reusable:
                raise RuntimeError(
                    f"{alias} already exists but cannot be safely skipped: "
                    + "; ".join(issues)
                )
            result = {
                "alias": alias,
                "profile": row["_profile"],
                "stage": row["_stage"],
                "status": "ALREADY_EXISTS_MATCH",
                "return_code": 0,
                "accession": str(existing.get("accession") or ""),
                "portal_url": portal_url(str(existing.get("accession") or "")),
                "portal_upload_status": str(existing.get("upload_status") or ""),
                "portal_object_status": str(existing.get("status") or ""),
                "log_path": "",
                "submitted_utc": now_utc(),
            }
            results.append(result)
            print(f"SKIP real upload; matching record already exists: {alias}")
            continue

        raw_result = run_iu_register(row, dry_run=False, ordinal=ordinal)
        obj = wait_for_record(alias)
        if obj is None:
            raw_result["status"] = "FAILED_NOT_FOUND_ON_PORTAL"
            results.append(raw_result)
            pd.DataFrame(results).to_csv(
                UPLOAD_WORK_DIR / f"{label}_submit_results.tsv",
                sep="\t",
                index=False,
                lineterminator="\n",
            )
            raise RuntimeError(
                f"{alias} was not found on the Portal after submission. "
                f"Review {raw_result['log_path']} before retrying."
            )

        issues = record_comparison(row, obj)
        accession = str(obj.get("accession") or "")
        result = {
            **raw_result,
            "accession": accession,
            "portal_url": portal_url(accession),
            "portal_upload_status": str(obj.get("upload_status") or ""),
            "portal_object_status": str(obj.get("status") or ""),
            "portal_comparison_issues": "; ".join(issues),
            "submitted_utc": now_utc(),
        }

        if issues:
            result["status"] = "PORTAL_RECORD_MISMATCH"
            results.append(result)
            pd.DataFrame(results).to_csv(
                UPLOAD_WORK_DIR / f"{label}_submit_results.tsv",
                sep="\t",
                index=False,
                lineterminator="\n",
            )
            raise RuntimeError(
                f"Portal record mismatch for {alias}: " + "; ".join(issues)
            )

        if raw_result["return_code"] != 0:
            result["status"] = "PORTAL_EXISTS_AFTER_CLIENT_ERROR_REVIEW_REQUIRED"
            results.append(result)
            pd.DataFrame(results).to_csv(
                UPLOAD_WORK_DIR / f"{label}_submit_results.tsv",
                sep="\t",
                index=False,
                lineterminator="\n",
            )
            raise RuntimeError(
                f"{alias} exists on the Portal, but iu_register exited nonzero. "
                f"Do not retry blindly; review {raw_result['log_path']} and the Portal record."
            )

        if not raw_result["aws_upload_success_message"]:
            result["status"] = "PORTAL_EXISTS_BUT_AWS_UPLOAD_NOT_CONFIRMED"
            results.append(result)
            pd.DataFrame(results).to_csv(
                UPLOAD_WORK_DIR / f"{label}_submit_results.tsv",
                sep="\t",
                index=False,
                lineterminator="\n",
            )
            raise RuntimeError(
                f"{alias} exists on the Portal, but the iu_register log did not contain "
                f"'AWS upload successful'. Do not retry blindly; review {raw_result['log_path']}."
            )

        result["status"] = "SUBMITTED_AND_PORTAL_MATCHED"
        results.append(result)
        print(f"VERIFIED: {alias} -> {accession}")
        time.sleep(SECONDS_BETWEEN_REAL_UPLOADS)

    output = pd.DataFrame(results)
    output_path = UPLOAD_WORK_DIR / f"{label}_submit_results.tsv"
    output.to_csv(output_path, sep="\t", index=False, lineterminator="\n")
    print(f"Submission results: {output_path}")
    print(output[["alias", "profile", "status", "accession", "portal_url"]].to_string(index=False))
    return output


def reconcile_rows(frame: pd.DataFrame, output_name: str) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for row in frame.to_dict("records"):
        alias = row["aliases"]
        obj = portal_get(alias)
        if not obj:
            rows.append({
                "day": row.get("_day", ""),
                "profile": row["_profile"],
                "alias": alias,
                "expected_file_set": normalize_identifier(row["file_set"]),
                "expected_md5sum": row["md5sum"],
                "expected_analysis_step_version": row.get("analysis_step_version", ""),
                "accession": "",
                "portal_url": "",
                "portal_file_set": "",
                "portal_md5sum": "",
                "portal_analysis_step_version": "",
                "portal_upload_status": "",
                "portal_object_status": "",
                "verification": "NOT_FOUND",
                "issues": "Portal record not found by alias",
                "submitted_file_name": row["submitted_file_name"],
            })
            continue
        issues = record_comparison(row, obj)
        accession = str(obj.get("accession") or "")
        rows.append({
            "day": row.get("_day", ""),
            "profile": row["_profile"],
            "alias": alias,
            "expected_file_set": normalize_identifier(row["file_set"]),
            "expected_md5sum": row["md5sum"],
            "expected_analysis_step_version": row.get("analysis_step_version", ""),
            "accession": accession,
            "portal_url": portal_url(accession),
            "portal_file_set": normalize_identifier(obj.get("file_set")),
            "portal_md5sum": str(obj.get("md5sum") or ""),
            "portal_analysis_step_version": normalize_identifier(
                obj.get("analysis_step_version")
            ),
            "portal_upload_status": str(obj.get("upload_status") or ""),
            "portal_object_status": str(obj.get("status") or ""),
            "verification": "MATCH" if not issues else "MISMATCH",
            "issues": "; ".join(issues),
            "submitted_file_name": row["submitted_file_name"],
        })
    output = pd.DataFrame(rows)
    path = METADATA_DIR / output_name
    output.to_csv(path, index=False, lineterminator="\n")
    print("Reconciliation:", path)
    print(output[[
        "day", "profile", "alias", "accession", "verification",
        "portal_upload_status", "portal_object_status",
    ]].to_string(index=False))
    return output

# ---------------------------------------------------------------------------
# ---------------------------------------------------------------------------
# Parent-object helpers
# ---------------------------------------------------------------------------

def normalize_repo_url(url: str) -> str:
    """Normalize a repository/source URL for conservative comparison."""
    text = str(url or "").strip().rstrip("/")
    if text.endswith(".git"):
        text = text[:-4]
    return text


def normalize_url_for_comparison(value: Any) -> str:
    return normalize_repo_url(str(value or "")).lower()


def normalize_link_list(value: Any) -> list[str]:
    if is_blank(value):
        return []
    values = value if isinstance(value, list) else split_identifiers(value)
    return [
        normalize_identifier(item)
        for item in values
        if normalize_identifier(item)
    ]


def parent_record_object(record: dict[str, Any]) -> dict[str, Any] | None:
    """Find a parent by alias or by the Portal's profile-specific unique key."""

    obj = portal_get(record["row"]["aliases"])
    if obj:
        return obj

    profile = record["profile"]
    row = record["row"]

    if profile == "software":
        return portal_find_software_by_name(row["name"])

    if profile == "software_version":
        return portal_find_software_version(
            row["version"],
            row["software"],
        )

    return None



def portal_link_matches(observed: Any, expected: str) -> bool:
    expected_norm = normalize_identifier(expected)
    observed_norm = normalize_identifier(observed)
    if observed_norm == expected_norm:
        return True

    if isinstance(observed, dict):
        if expected in normalize_aliases(observed.get("aliases")):
            return True
        for field in ("accession", "name", "uuid", "@id"):
            if normalize_identifier(observed.get(field)) == expected_norm:
                return True

    if observed_norm:
        try:
            linked = portal_get(observed_norm)
        except Exception:
            linked = None
        if linked:
            if expected in normalize_aliases(linked.get("aliases")):
                return True
            for field in ("accession", "name", "uuid", "@id"):
                if normalize_identifier(linked.get(field)) == expected_norm:
                    return True

    # Software names are identifying properties but are not always directly
    # routable as /<name>/. Fall back to an exact Software search.
    try:
        expected_software = portal_find_software_by_name(expected_norm)
    except Exception:
        expected_software = None
    if expected_software:
        if isinstance(observed, dict):
            observed_object = observed
        elif observed_norm:
            try:
                observed_object = portal_get(observed_norm)
            except Exception:
                observed_object = None
        else:
            observed_object = None
        if observed_object and (
            portal_object_identifiers(expected_software)
            & portal_object_identifiers(observed_object)
        ):
            return True
    return False


def compare_parent_record(
    record: dict[str, Any],
    obj: dict[str, Any],
) -> list[str]:
    row = record["row"]
    profile = record["profile"]
    issues: list[str] = []

    aliases = normalize_aliases(obj.get("aliases"))
    if (
        row["aliases"] not in aliases
        and profile not in {"software", "software_version"}
    ):
        issues.append(f"expected alias absent: {row['aliases']}")

    if profile == "software":
        if str(obj.get("name") or "") != row["name"]:
            issues.append(
                f"name mismatch: expected={row['name']}, portal={obj.get('name')}"
            )

    elif profile == "software_version":
        if str(obj.get("version") or "") != row["version"]:
            issues.append(
                f"version mismatch: expected={row['version']}, "
                f"portal={obj.get('version')}"
            )
        expected_software = row["software"]
        observed_software = obj.get("software")
        if observed_software and not software_link_matches(
            observed_software,
            expected_software,
        ):
            issues.append(
                f"software mismatch: expected={expected_software}, "
                f"portal={normalize_identifier(observed_software)}"
            )

    elif profile == "analysis_step":
        if str(obj.get("step_label") or "") != row["step_label"]:
            issues.append(
                f"step_label mismatch: expected={row['step_label']}, "
                f"portal={obj.get('step_label')}"
            )

    elif profile == "analysis_step_version":
        expected_step = row["analysis_step"]
        observed_step = obj.get("analysis_step")
        if observed_step and not portal_link_matches(
            observed_step,
            expected_step,
        ):
            issues.append(
                f"analysis_step mismatch: expected={expected_step}, "
                f"portal={normalize_identifier(observed_step)}"
            )

        expected_versions = split_identifiers(row["software_versions"])
        observed_versions = obj.get("software_versions") or []
        if not isinstance(observed_versions, list):
            observed_versions = [observed_versions]
        unmatched_versions = [
            expected
            for expected in expected_versions
            if not any(
                portal_link_matches(observed, expected)
                for observed in observed_versions
            )
        ]
        if observed_versions and unmatched_versions:
            issues.append(
                "software_versions mismatch: expected links not found="
                f"{unmatched_versions}; portal="
                f"{normalize_link_list(observed_versions)}"
            )

    status = str(obj.get("status") or "").lower()
    if status in {"revoked", "archived", "deleted"}:
        issues.append(f"Portal object status is {status!r}")

    return issues


def parent_status_table(
    records: list[dict[str, Any]],
) -> pd.DataFrame:
    refresh_parent_links(records)
    rows: list[dict[str, Any]] = []
    for record in records:
        obj = parent_record_object(record)
        issues = compare_parent_record(record, obj) if obj else []
        accession = str(obj.get("accession") or "") if obj else ""
        rows.append({
            "key": record["key"],
            "profile": record["profile"],
            "alias": record["row"]["aliases"],
            "found": bool(obj),
            "accession": accession,
            "portal_url": portal_url(accession),
            "status": str(obj.get("status") or "") if obj else "",
            "verification": (
                "MATCH" if obj and not issues
                else "MISMATCH" if obj
                else "NOT_FOUND"
            ),
            "issues": "; ".join(issues),
            "depends_on": ",".join(record.get("depends_on", [])),
        })
    return pd.DataFrame(rows)


def parent_run_result_passed(result: dict[str, Any], *, dry_run: bool) -> bool:
    if result["return_code"] != 0 or result["error_pattern_found"]:
        return False
    if dry_run and not result["dry_run_marker"]:
        return False
    return True


def write_profile_tsv(
    records: list[dict[str, Any]],
    profile: str,
    path: Path,
) -> None:
    subset = [record["row"] for record in records if record["profile"] == profile]
    if not subset:
        return
    field_order: list[str] = []
    for row in subset:
        for key in row:
            if key.startswith("_"):
                continue
            if key not in field_order:
                field_order.append(key)
    frame = pd.DataFrame([
        {key: row.get(key, "") for key in field_order}
        for row in subset
    ])
    frame.to_csv(path, sep="\t", index=False, lineterminator="\n")


def wait_for_parent_record(
    record: dict[str, Any],
    *,
    attempts: int = 12,
    wait_seconds: float = 5.0,
) -> dict[str, Any] | None:
    for attempt in range(1, attempts + 1):
        obj = parent_record_object(record)
        if obj:
            return obj
        if attempt < attempts:
            time.sleep(wait_seconds)
    return None

PARENT_LAYERS = [
    ["preprocessing_software", "perturbnmf_software"],
    ["preprocessing_software_version", "perturbnmf_software_version"],
    ["preprocessing_analysis_step"],
    ["perturbnmf_analysis_step"],
    [
        "preprocessing_analysis_step_version",
        "perturbnmf_analysis_step_version",
    ],
]


def parent_record_map(
    records: list[dict[str, Any]],
) -> dict[str, dict[str, Any]]:
    return {record["key"]: record for record in records}


def dry_run_first_missing_parent_layer(
    records: list[dict[str, Any]],
) -> pd.DataFrame:
    """Dry-run the earliest missing dependency layer that can be validated."""

    refresh_parent_links(records)
    record_map = parent_record_map(records)
    status = parent_status_table(records)
    if status["verification"].eq("MISMATCH").any():
        bad = status.loc[
            status["verification"].eq("MISMATCH"),
            ["key", "alias", "issues"],
        ]
        raise RuntimeError(
            "Existing parent metadata mismatch; do not POST or PATCH automatically:\n"
            + bad.to_string(index=False)
        )

    found_by_key = dict(zip(status["key"], status["found"]))
    results: list[dict[str, Any]] = []

    for layer_number, layer in enumerate(PARENT_LAYERS, start=1):
        missing = [key for key in layer if not found_by_key.get(key, False)]
        if not missing:
            continue

        for key in missing:
            record = record_map[key]
            unmet = [
                dependency
                for dependency in record.get("depends_on", [])
                if not found_by_key.get(dependency, False)
            ]
            if unmet:
                results.append({
                    "key": key,
                    "profile": record["profile"],
                    "alias": record["row"]["aliases"],
                    "layer": layer_number,
                    "status": "BLOCKED_BY_MISSING_PARENT",
                    "details": ",".join(unmet),
                    "log_path": "",
                    "row_tsv": "",
                })
                continue

            result = run_iu_register(
                record["row"],
                dry_run=True,
                ordinal=100 + layer_number,
            )
            passed = parent_run_result_passed(result, dry_run=True)
            results.append({
                "key": key,
                "profile": record["profile"],
                "alias": record["row"]["aliases"],
                "layer": layer_number,
                "status": "DRY_RUN_PASS" if passed else "DRY_RUN_FAIL",
                "details": (
                    ""
                    if passed
                    else (
                        f"return_code={result['return_code']}; "
                        f"error_pattern_found={result['error_pattern_found']}; "
                        f"dry_run_marker={result['dry_run_marker']}"
                    )
                ),
                "log_path": result["log_path"],
                "row_tsv": result["row_tsv"],
            })

        # Stop after the first layer containing missing objects. Later layers
        # refer to objects that do not yet exist in the production database.
        break

    if not results:
        results.append({
            "key": "",
            "profile": "",
            "alias": "",
            "layer": "",
            "status": "ALL_PARENT_OBJECTS_ALREADY_EXIST",
            "details": "",
            "log_path": "",
            "row_tsv": "",
        })

    frame = pd.DataFrame(results)
    path = PARENT_METADATA_DIR / "parent_first_missing_layer_dryrun.tsv"
    frame.to_csv(path, sep="\t", index=False, lineterminator="\n")
    print(frame.to_string(index=False))
    print("\nParent dry-run report:", path)
    return frame


def submit_all_missing_parent_objects(
    records: list[dict[str, Any]],
) -> pd.DataFrame:
    """Dry-run and production-POST each missing parent in dependency order."""

    refresh_parent_links(records)
    initial_status = parent_status_table(records)
    if initial_status["verification"].eq("MISMATCH").any():
        raise RuntimeError(
            "Existing parent metadata mismatch; no production writes were attempted."
        )
    if initial_status["verification"].eq("MATCH").all():
        print("All parent objects already exist and match; no production POST is needed.")
        return initial_status.assign(
            layer="",
            status="ALREADY_EXISTS_MATCH",
            dryrun_log="",
            submit_log="",
        )

    if PARENT_REAL_CONFIRMATION != EXPECTED_PARENT_TOKEN:
        raise RuntimeError(
            "Production parent creation is blocked. Review the parent status "
            "and dry-run logs, then set PARENT_REAL_CONFIRMATION exactly to "
            f"{EXPECTED_PARENT_TOKEN!r} in the configuration cell and rerun it."
        )

    record_map = parent_record_map(records)
    results: list[dict[str, Any]] = []
    ordinal = 200

    for layer_number, layer in enumerate(PARENT_LAYERS, start=1):
        for key in layer:
            refresh_parent_links(records)
            record = record_map[key]
            existing = parent_record_object(record)
            if existing:
                issues = compare_parent_record(record, existing)
                if issues:
                    raise RuntimeError(
                        f"Existing parent object mismatch for {key}: "
                        + "; ".join(issues)
                    )
                results.append({
                    "key": key,
                    "profile": record["profile"],
                    "alias": record["row"]["aliases"],
                    "layer": layer_number,
                    "status": "ALREADY_EXISTS_MATCH",
                    "accession": str(existing.get("accession") or ""),
                    "portal_url": portal_url(str(existing.get("accession") or "")),
                    "dryrun_log": "",
                    "submit_log": "",
                })
                continue

            unmet: list[str] = []
            for dependency in record.get("depends_on", []):
                dependency_record = record_map[dependency]
                dependency_obj = parent_record_object(dependency_record)
                if not dependency_obj:
                    unmet.append(dependency)
            if unmet:
                raise RuntimeError(
                    f"Cannot create {key}; missing dependencies: {unmet}"
                )

            refresh_parent_links(records)
            ordinal += 1
            dry_result = run_iu_register(
                record["row"],
                dry_run=True,
                ordinal=ordinal,
            )
            if not parent_run_result_passed(dry_result, dry_run=True):
                raise RuntimeError(
                    f"Parent dry run failed for {key}; review "
                    f"{dry_result['log_path']}"
                )

            ordinal += 1
            submit_result = run_iu_register(
                record["row"],
                dry_run=False,
                ordinal=ordinal,
            )

            created = wait_for_parent_record(record)
            if not created:
                raise RuntimeError(
                    f"Production POST did not yield a queryable parent record "
                    f"for {key}. Review {submit_result['log_path']} before retrying."
                )

            refresh_parent_links(records)
            issues = compare_parent_record(record, created)
            if issues:
                raise RuntimeError(
                    f"Portal object created/retrieved for {key}, but metadata "
                    f"does not match: {'; '.join(issues)}"
                )

            if (
                submit_result["return_code"] != 0
                or submit_result["error_pattern_found"]
            ):
                print(
                    "WARNING:",
                    key,
                    "was found and matched after a client-side error. "
                    "Do not retry this object.",
                )

            accession = str(created.get("accession") or "")
            results.append({
                "key": key,
                "profile": record["profile"],
                "alias": record["row"]["aliases"],
                "layer": layer_number,
                "status": "SUBMITTED_AND_PORTAL_MATCHED",
                "accession": accession,
                "portal_url": portal_url(accession),
                "dryrun_log": dry_result["log_path"],
                "submit_log": submit_result["log_path"],
            })
            time.sleep(SECONDS_BETWEEN_REAL_UPLOADS)

    frame = pd.DataFrame(results)
    path = PARENT_METADATA_DIR / "parent_submission_results.tsv"
    frame.to_csv(path, sep="\t", index=False, lineterminator="\n")
    print(frame.to_string(index=False))
    print("\nParent submission results:", path)
    return frame

## 1. Set and test IGVF credentials

Credentials are entered with hidden prompts and remain only in the current notebook kernel.

In [3]:
if not os.environ.get("IGVF_API_KEY"):
    os.environ["IGVF_API_KEY"] = getpass.getpass("IGVF_API_KEY: ")
if not os.environ.get("IGVF_SECRET_KEY"):
    os.environ["IGVF_SECRET_KEY"] = getpass.getpass("IGVF_SECRET_KEY: ")

for name in ("IGVF_API_KEY", "IGVF_SECRET_KEY"):
    value = os.environ.get(name, "")
    print(
        f"{name}: {'SET' if value else 'MISSING'}, "
        f"length={len(value)}, whitespace={value != value.strip()}"
    )

# Authenticated read-only check against one active AnalysisSet.
auth_test = portal_get(ANALYSIS_SETS["D0"])
if not auth_test:
    raise RuntimeError("Authentication test did not return the D0 AnalysisSet")
print(
    "Authenticated Portal access confirmed:",
    auth_test.get("accession"),
    auth_test.get("@type"),
)

IGVF_API_KEY: SET, length=8, whitespace=False
IGVF_SECRET_KEY: SET, length=16, whitespace=False
Authenticated Portal access confirmed: IGVFDS0463XTUH ['AnalysisSet', 'FileSet', 'Item']


## 2. Use fixed first-pass software versions

This notebook does not clone, fetch, or query either GitHub repository.

The fixed values are:

```text
CC-Perturb-seq 300-gene preprocessing: v0.1.0
PerturbNMF:                            v0.1.0
```

For each SoftwareVersion, the repository root URL is recorded as `source_url`.
The generated provenance report explicitly records that the version is
hard-coded and that no exact commit or tag was resolved.

This is intended to unblock a first-pass IGVF upload. Reconcile the
SoftwareVersion records with the exact commit or tag before final release if
DACC requests immutable provenance.


In [4]:
resolution_rows = []

for key, spec in HARD_CODED_SOFTWARE_VERSIONS.items():
    portal_version = str(spec["portal_version"]).strip()
    if not PORTAL_VERSION_PATTERN.fullmatch(portal_version):
        raise ValueError(
            f"Invalid hard-coded IGVF SoftwareVersion for {key}: "
            f"{portal_version!r}"
        )

    resolution_rows.append({
        "repository_key": key,
        "title": spec["title"],
        "software_name": spec["software_name"],
        "software_alias_base": spec["software_alias_base"],
        "portal_version": portal_version,
        "version_source": spec["version_source"],
        "software_source_url": spec["software_source_url"],
        "software_version_source_url": spec["software_version_source_url"],
        "provenance_note": spec["provenance_note"],
        # Deliberately blank: this first-pass notebook performs no Git access
        # and makes no exact commit/tag claim.
        "commit_sha": "",
        "short_sha": "",
        "commit_date": "",
        "default_branch": "",
        "exact_tag": "",
        "declared_version_raw": "",
        "declared_version_source": "",
        "commit_url": "",
        "archive_url": "",
    })

software_resolution = (
    pd.DataFrame(resolution_rows)
    .set_index("repository_key", drop=False)
    .sort_index()
)

resolution_path = UPLOAD_WORK_DIR / "software_version_resolution.tsv"
software_resolution.to_csv(
    resolution_path,
    sep="\t",
    index=False,
    lineterminator="\n",
)

print("\nHard-coded first-pass software versions:")
print(
    software_resolution[[
        "repository_key",
        "portal_version",
        "version_source",
        "software_version_source_url",
        "provenance_note",
    ]].to_string(index=False)
)
print("\nVersion-resolution report:", resolution_path)
print(
    "\nNOTE: no Git clone/fetch/API call was performed and no exact "
    "commit is claimed."
)

prep = software_resolution.loc["preprocessing"].to_dict()
pnmf = software_resolution.loc["perturbnmf"].to_dict()

PREPROCESS_SOFTWARE_ALIAS = "jesse-engreitz:software_CC-Perturb-seq"
PERTURBNMF_SOFTWARE_ALIAS = "jesse-engreitz:software_PerturbNMF"

PREPROCESS_SOFTWARE_VERSION_ALIAS = (
    "jesse-engreitz:software_version_CC-Perturb-seq_"
    f"{prep['portal_version']}"
)
PERTURBNMF_SOFTWARE_VERSION_ALIAS = (
    "jesse-engreitz:software_version_PerturbNMF_"
    f"{pnmf['portal_version']}"
)

PREPROCESS_ANALYSIS_STEP_ALIAS = (
    "jesse-engreitz:"
    "analysis_step_WTC11-EC_300gene_CC-Perturbseq_preprocessing"
)
PERTURBNMF_ANALYSIS_STEP_ALIAS = (
    "jesse-engreitz:"
    "analysis_step_WTC11-EC_300gene_PerturbNMF_gene_programs"
)

PREPROCESS_ANALYSIS_STEP_VERSION_ALIAS = (
    "jesse-engreitz:"
    "analysis_step_version_WTC11-EC_300gene_"
    "CC-Perturbseq_preprocessing_"
    f"{prep['portal_version']}"
)
PERTURBNMF_ANALYSIS_STEP_VERSION_ALIAS = (
    "jesse-engreitz:"
    "analysis_step_version_WTC11-EC_300gene_"
    "PerturbNMF_gene_programs_"
    f"{pnmf['portal_version']}"
)

# Parent objects needed by the 16 analysis files. Records are ordered by
# dependency: Software -> SoftwareVersion -> AnalysisStep -> AnalysisStepVersion.
PARENT_RECORDS = [
    {
        "key": "preprocessing_software",
        "profile": "software",
        "depends_on": [],
        "row": {
            "aliases": PREPROCESS_SOFTWARE_ALIAS,
            "award": AWARD,
            "lab": LAB,
            "name": prep["software_name"],
            "title": prep["title"],
            "description": (
                "CC-Perturb-seq preprocessing code used for the WTC11 "
                "300-gene CC-Perturb-seq dataset, including gene and guide "
                "counting, guide assignment, matrix generation, and QC."
            ),
            "source_url": prep["software_source_url"],
            "_profile": "software",
            "_stage": "parent",
        },
    },
    {
        "key": "perturbnmf_software",
        "profile": "software",
        "depends_on": [],
        "row": {
            "aliases": PERTURBNMF_SOFTWARE_ALIAS,
            "award": AWARD,
            "lab": LAB,
            "name": pnmf["software_name"],
            "title": pnmf["title"],
            "description": (
                "PerturbNMF pipeline used for cNMF gene-program inference, "
                "program evaluation, perturbation calibration, and "
                "interpretation of single-cell perturbation data."
            ),
            "source_url": pnmf["software_source_url"],
            "_profile": "software",
            "_stage": "parent",
        },
    },
    {
        "key": "preprocessing_software_version",
        "profile": "software_version",
        "depends_on": ["preprocessing_software"],
        "row": {
            "aliases": PREPROCESS_SOFTWARE_VERSION_ALIAS,
            "award": AWARD,
            "lab": LAB,
            "software": prep["software_name"],
            "version": prep["portal_version"],
            "source_url": prep["software_version_source_url"],
            "description": (
                f"First-pass hard-coded semantic version "
                f"{prep['portal_version']} for the WTC11 300-gene "
                "preprocessing workflow. Repository URL is recorded; exact "
                "tag/commit was not resolved in this upload notebook."
            ),
            "_profile": "software_version",
            "_stage": "parent",
        },
    },
    {
        "key": "perturbnmf_software_version",
        "profile": "software_version",
        "depends_on": ["perturbnmf_software"],
        "row": {
            "aliases": PERTURBNMF_SOFTWARE_VERSION_ALIAS,
            "award": AWARD,
            "lab": LAB,
            "software": pnmf["software_name"],
            "version": pnmf["portal_version"],
            "source_url": pnmf["software_version_source_url"],
            "description": (
                f"First-pass hard-coded semantic version "
                f"{pnmf['portal_version']} for PerturbNMF. Repository URL "
                "is recorded; exact tag/commit was not resolved in this "
                "upload notebook."
            ),
            "_profile": "software_version",
            "_stage": "parent",
        },
    },
    {
        "key": "preprocessing_analysis_step",
        "profile": "analysis_step",
        "depends_on": [],
        "row": {
            "aliases": PREPROCESS_ANALYSIS_STEP_ALIAS,
            "award": AWARD,
            "lab": LAB,
            "title": (
                "Perturb-seq preprocessing, guide assignment, and matrix "
                "generation step"
            ),
            "step_label": (
                "perturbseq-preprocessing-guide-assignment-matrix-step"
            ),
            "analysis_step_types": (
                "barcode correction, cell to feature barcode mapping, "
                "filtering, matrix generation, quantification"
            ),
            "input_content_types": "reads",
            "output_content_types": (
                "annotated cell by gene and guide matrix, "
                "guide quantifications, per-cell quality report"
            ),
            "description": (
                "Processes raw GEX and guide FASTQs and assay metadata to "
                "produce filtered annotated cell-by-gene-and-guide matrices, "
                "guide assignments, guide quantifications, and per-cell QC."
            ),
            "_profile": "analysis_step",
            "_stage": "parent",
        },
    },
    {
        "key": "perturbnmf_analysis_step",
        "profile": "analysis_step",
        "depends_on": ["preprocessing_analysis_step"],
        "row": {
            "aliases": PERTURBNMF_ANALYSIS_STEP_ALIAS,
            "award": AWARD,
            "lab": LAB,
            "title": (
                "PerturbNMF cNMF gene program analysis and evaluation step"
            ),
            "step_label": "perturbnmf-gene-program-inference-step",
            "analysis_step_types": (
                "counts normalization, matrix generation, quantification"
            ),
            "input_content_types": (
                "annotated cell by gene and guide matrix"
            ),
            "output_content_types": (
                "cell by gene program matrix, gene programs, "
                "gene program regulators, gene universe, "
                "regulator-regulator correlation, "
                "program-program correlation"
            ),
            "parents": PREPROCESS_ANALYSIS_STEP_ALIAS,
            "description": (
                "Uses the processed single-cell matrix to infer cNMF gene "
                "programs and evaluate perturbation-program relationships, "
                "program quality, and program/regulator correlations."
            ),
            "_profile": "analysis_step",
            "_stage": "parent",
        },
    },
    {
        "key": "preprocessing_analysis_step_version",
        "profile": "analysis_step_version",
        "depends_on": [
            "preprocessing_software_version",
            "preprocessing_analysis_step",
        ],
        "row": {
            "aliases": PREPROCESS_ANALYSIS_STEP_VERSION_ALIAS,
            "award": AWARD,
            "lab": LAB,
            "analysis_step": PREPROCESS_ANALYSIS_STEP_ALIAS,
            "software_versions": PREPROCESS_SOFTWARE_VERSION_ALIAS,
            "description": (
                "First-pass preprocessing step using "
                f"{prep['title']} {prep['portal_version']}. Exact Git "
                "tag/commit was not resolved in this upload notebook."
            ),
            "_profile": "analysis_step_version",
            "_stage": "parent",
        },
    },
    {
        "key": "perturbnmf_analysis_step_version",
        "profile": "analysis_step_version",
        "depends_on": [
            "perturbnmf_software_version",
            "perturbnmf_analysis_step",
        ],
        "row": {
            "aliases": PERTURBNMF_ANALYSIS_STEP_VERSION_ALIAS,
            "award": AWARD,
            "lab": LAB,
            "analysis_step": PERTURBNMF_ANALYSIS_STEP_ALIAS,
            "software_versions": PERTURBNMF_SOFTWARE_VERSION_ALIAS,
            "description": (
                "First-pass PerturbNMF gene-program step using "
                f"{pnmf['title']} {pnmf['portal_version']}. Exact Git "
                "tag/commit was not resolved in this upload notebook."
            ),
            "_profile": "analysis_step_version",
            "_stage": "parent",
        },
    },
]

# Export reusable parent-object metadata.
for profile, filename in [
    ("software", "01_software.tsv"),
    ("software_version", "02_software_version.tsv"),
    ("analysis_step", "03_analysis_step.tsv"),
    ("analysis_step_version", "04_analysis_step_version.tsv"),
]:
    write_profile_tsv(
        PARENT_RECORDS,
        profile,
        PARENT_METADATA_DIR / filename,
    )

# Make resolved copies of the file metadata. The source TSVs remain unchanged.
for path in (
    SOURCE_MASTER_MANIFEST,
    SOURCE_H5AD_TSV,
    SOURCE_H5MU_TSV,
    SOURCE_TABULAR_TSV,
):
    if not path.is_file():
        raise FileNotFoundError(f"Missing source metadata: {path}")

source_manifest = pd.read_csv(
    SOURCE_MASTER_MANIFEST,
    sep="\t",
    dtype=str,
    keep_default_na=False,
)
source_h5ad = pd.read_csv(
    SOURCE_H5AD_TSV,
    sep="\t",
    dtype=str,
    keep_default_na=False,
)
source_h5mu = pd.read_csv(
    SOURCE_H5MU_TSV,
    sep="\t",
    dtype=str,
    keep_default_na=False,
)
source_tabular = pd.read_csv(
    SOURCE_TABULAR_TSV,
    sep="\t",
    dtype=str,
    keep_default_na=False,
)

resolved_manifest = source_manifest.copy()
resolved_manifest.loc[
    resolved_manifest["file_format"].eq("h5mu"),
    "analysis_step_version",
] = PERTURBNMF_ANALYSIS_STEP_VERSION_ALIAS
resolved_manifest.loc[
    ~resolved_manifest["file_format"].eq("h5mu"),
    "analysis_step_version",
] = PREPROCESS_ANALYSIS_STEP_VERSION_ALIAS

resolved_h5ad = source_h5ad.copy()
resolved_h5ad["analysis_step_version"] = (
    PREPROCESS_ANALYSIS_STEP_VERSION_ALIAS
)
resolved_h5mu = source_h5mu.copy()
resolved_h5mu["analysis_step_version"] = (
    PERTURBNMF_ANALYSIS_STEP_VERSION_ALIAS
)
resolved_tabular = source_tabular.copy()
resolved_tabular["analysis_step_version"] = (
    PREPROCESS_ANALYSIS_STEP_VERSION_ALIAS
)

resolved_manifest.to_csv(
    MASTER_MANIFEST,
    sep="\t",
    index=False,
    lineterminator="\n",
)
resolved_h5ad.to_csv(
    H5AD_TSV,
    sep="\t",
    index=False,
    lineterminator="\n",
)
resolved_h5mu.to_csv(
    H5MU_TSV,
    sep="\t",
    index=False,
    lineterminator="\n",
)
resolved_tabular.to_csv(
    TABULAR_TSV,
    sep="\t",
    index=False,
    lineterminator="\n",
)

provenance_report = {
    "resolved_utc": now_utc(),
    "policy": (
        "Hard-coded first-pass semantic versions were used. No Git clone, "
        "fetch, GitHub API call, exact tag lookup, or commit lookup was "
        "performed. Repository root URLs are recorded as source_url."
    ),
    "requires_final_commit_reconciliation": True,
    "preprocessing": prep,
    "perturbnmf": pnmf,
    "aliases": {
        "preprocessing_software": PREPROCESS_SOFTWARE_ALIAS,
        "perturbnmf_software": PERTURBNMF_SOFTWARE_ALIAS,
        "preprocessing_software_version": PREPROCESS_SOFTWARE_VERSION_ALIAS,
        "perturbnmf_software_version": PERTURBNMF_SOFTWARE_VERSION_ALIAS,
        "preprocessing_analysis_step": PREPROCESS_ANALYSIS_STEP_ALIAS,
        "perturbnmf_analysis_step": PERTURBNMF_ANALYSIS_STEP_ALIAS,
        "preprocessing_analysis_step_version": (
            PREPROCESS_ANALYSIS_STEP_VERSION_ALIAS
        ),
        "perturbnmf_analysis_step_version": (
            PERTURBNMF_ANALYSIS_STEP_VERSION_ALIAS
        ),
    },
}
provenance_path = UPLOAD_WORK_DIR / "resolved_software_provenance.json"
provenance_path.write_text(
    json.dumps(provenance_report, indent=2) + "\n",
    encoding="utf-8",
)

print("\nResolved AnalysisStepVersion aliases:")
print("  preprocessing:", PREPROCESS_ANALYSIS_STEP_VERSION_ALIAS)
print("  PerturbNMF:   ", PERTURBNMF_ANALYSIS_STEP_VERSION_ALIAS)
print("\nResolved file metadata:")
for path in (MASTER_MANIFEST, H5AD_TSV, H5MU_TSV, TABULAR_TSV):
    print(" ", path)
print("Provenance JSON:", provenance_path)



Hard-coded first-pass software versions:
repository_key portal_version       version_source                                          software_version_source_url                                                                                                    provenance_note
    perturbnmf         v0.1.0 hardcoded_first_pass                            https://github.com/EngreitzLab/PerturbNMF Hard-coded semantic version v0.1.0 for first-pass IGVF submission; exact tag/commit not recorded in this notebook.
 preprocessing         v0.1.0 hardcoded_first_pass https://github.com/EngreitzLab/CC-Perturb-seq_300genes_preprocessing Hard-coded semantic version v0.1.0 for first-pass IGVF submission; exact tag/commit not recorded in this notebook.

Version-resolution report: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/_upload_work/software_version_resolution.tsv

NOTE: no Git clone/fetch/API call was performed and no exact commit is claimed.

Resolved An

## 3. Local preflight

Revalidate the exact final files and MD5 checksums.

In [5]:
required_paths = [
    READY_MARKER,
    MASTER_MANIFEST,
    H5AD_TSV,
    H5MU_TSV,
    TABULAR_TSV,
    IGVF_PYTHON,
    IU_REGISTER,
]
missing = [str(path) for path in required_paths if not path.is_file()]
if missing:
    raise FileNotFoundError("Required files are missing:\n" + "\n".join(missing))

print(READY_MARKER.read_text(encoding="utf-8"))

manifest = pd.read_csv(MASTER_MANIFEST, sep="\t", dtype=str, keep_default_na=False)
if len(manifest) != 16:
    raise AssertionError(f"Expected 16 rows in the master manifest, found {len(manifest)}")
if not manifest["validation_status"].eq("PASS").all():
    raise RuntimeError("At least one master-manifest row is not locally validated PASS")
if manifest["alias"].duplicated().any():
    raise RuntimeError("Duplicate aliases found in the master manifest")
if manifest["submitted_file_name"].duplicated().any():
    raise RuntimeError("Duplicate file paths found in the master manifest")

alias_to_day = dict(zip(manifest["alias"], manifest["day"]))

h5ad = load_metadata(H5AD_TSV, "matrix_file", "h5ad")
h5mu = load_metadata(H5MU_TSV, "matrix_file", "h5mu")
tabular = load_metadata(TABULAR_TSV, "tabular_file", "tabular")

if (len(h5ad), len(h5mu), len(tabular)) != (4, 4, 8):
    raise AssertionError(
        f"Expected 4 h5ad, 4 h5mu, and 8 tabular rows; found "
        f"{len(h5ad)}, {len(h5mu)}, and {len(tabular)}"
    )

all_rows = pd.concat([h5ad, h5mu, tabular], ignore_index=True)
all_rows["_day"] = all_rows["aliases"].map(alias_to_day)
if all_rows["_day"].isna().any():
    bad = all_rows.loc[all_rows["_day"].isna(), "aliases"].tolist()
    raise RuntimeError(f"Metadata aliases absent from master manifest: {bad}")

if all_rows["aliases"].duplicated().any():
    raise RuntimeError("Duplicate aliases found across metadata TSVs")
if all_rows["submitted_file_name"].duplicated().any():
    raise RuntimeError("Duplicate submitted_file_name values found across metadata TSVs")

local_rows = []
for index, row in enumerate(all_rows.to_dict("records"), start=1):
    path = Path(row["submitted_file_name"])
    day = row["_day"]
    expected_set = ANALYSIS_SETS[day]
    if normalize_identifier(row["file_set"]) != expected_set:
        raise RuntimeError(
            f"Wrong AnalysisSet for {row['aliases']}: "
            f"expected {expected_set}, found {row['file_set']}"
        )
    if not path.is_file():
        raise FileNotFoundError(f"Missing final upload file: {path}")
    if path.stat().st_size <= 0:
        raise RuntimeError(f"Empty final upload file: {path}")

    observed_md5 = md5sum(path) if VERIFY_MD5_BEFORE_UPLOAD else row["md5sum"]
    md5_match = observed_md5 == row["md5sum"]
    print(
        f"[{index:02d}/16] {day} {row['_profile']} "
        f"{path.name}: size={path.stat().st_size:,}, md5_match={md5_match}"
    )
    local_rows.append({
        "day": day,
        "profile": row["_profile"],
        "stage": row["_stage"],
        "alias": row["aliases"],
        "file_set": expected_set,
        "submitted_file_name": str(path),
        "size_bytes": path.stat().st_size,
        "expected_md5sum": row["md5sum"],
        "observed_md5sum": observed_md5,
        "md5_match": md5_match,
    })

local_preflight = pd.DataFrame(local_rows)
local_preflight_path = UPLOAD_WORK_DIR / "local_preflight_16.tsv"
local_preflight.to_csv(
    local_preflight_path,
    sep="\t",
    index=False,
    lineterminator="\n",
)
if not local_preflight["md5_match"].all():
    raise RuntimeError(f"MD5 mismatch detected. Review {local_preflight_path}")

print("\nLocal preflight passed for all 16 files")
print("Report:", local_preflight_path)
print(all_rows.groupby(["_day", "_stage"]).size().to_string())

CC-Perturb-seq IGVF analysis-file build: READY
generated_utc: 2026-08-11T09:31:13+00:00
validated_files: 16
h5ad_files: 4
h5mu_files: 4
guide_quantification_files: 4
per_cell_quality_report_files: 4
master_manifest: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/metadata/upload_manifest_16_files.tsv
validation_report: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/_prep_work/file_validation_16.tsv
h5ad_metadata: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/metadata/01_matrix_file_h5ad.tsv
h5mu_metadata: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/metadata/02_matrix_file_h5mu.tsv
tabular_metadata: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/metadata/03_tabular_file_step1.tsv
column_dictionary: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/doc

## 4. Reconcile the required parent objects

The 16 files require two parent chains:

- CC-Perturb-seq preprocessing:
  `Software -> SoftwareVersion -> AnalysisStep -> AnalysisStepVersion`
- PerturbNMF:
  `Software -> SoftwareVersion -> AnalysisStep -> AnalysisStepVersion`

This section is read-only. It queries aliases and software names and stops on
any metadata conflict rather than PATCHing an existing record.

In [6]:
parent_status = parent_status_table(PARENT_RECORDS)
parent_status_path = PARENT_METADATA_DIR / "parent_object_status.tsv"
parent_status.to_csv(
    parent_status_path,
    sep="\t",
    index=False,
    lineterminator="\n",
)

print(parent_status[[
    "key", "profile", "alias", "verification", "accession", "issues"
]].to_string(index=False))
print("\nParent status report:", parent_status_path)

if parent_status["verification"].eq("MISMATCH").any():
    raise RuntimeError(
        "At least one existing parent object conflicts with the resolved "
        "repository version or planned metadata. Review the report; the "
        "notebook will not PATCH parent objects automatically."
    )

print("\nParent metadata TSVs:")
for path in sorted(PARENT_METADATA_DIR.glob("0*_*.tsv")):
    print(" ", path)

                                key               profile                                                                                    alias verification accession issues
             preprocessing_software              software                                                   jesse-engreitz:software_CC-Perturb-seq        MATCH                 
                perturbnmf_software              software                                                       jesse-engreitz:software_PerturbNMF        MATCH                 
     preprocessing_software_version      software_version                                    jesse-engreitz:software_version_CC-Perturb-seq_v0.1.0    NOT_FOUND                 
        perturbnmf_software_version      software_version                                        jesse-engreitz:software_version_PerturbNMF_v0.1.0    NOT_FOUND                 
        preprocessing_analysis_step         analysis_step                jesse-engreitz:analysis_step_WTC11-EC_300g

## 5. Dry-run the earliest missing parent layer

Only the earliest missing dependency layer is dry-run. For example, a missing
SoftwareVersion cannot be validated until its Software exists in production.
Review the generated one-row TSV and log before authorizing production parent
creation.

In [7]:
parent_dryrun = dry_run_first_missing_parent_layer(PARENT_RECORDS)
if parent_dryrun["status"].eq("DRY_RUN_FAIL").any():
    raise RuntimeError(
        "At least one parent dry run failed. Review the log paths above."
    )

if parent_dryrun["status"].eq("ALL_PARENT_OBJECTS_ALREADY_EXIST").all():
    print("All required parent objects already exist and match.")
else:
    print(
        "\nOnly the earliest missing dependency layer was dry-run. "
        "Later layers cannot be validated against the production database "
        "until their parents exist. Review these logs before authorizing "
        "the production parent-object cell."
    )

Command: /oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/bin/python /oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/igvf_utils/igvf_utils/MetaDataRegistration/iu_register.py -m prod -p software_version -d -i /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/_upload_work/one_row_tsv/102_software_version_jesse-engreitz_software_version_CC-Perturb-seq_v0.1.0.tsv
Log: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/logs/igvf_utils_analysis_upload/102_software_version_jesse-engreitz_software_version_CC-Perturb-seq_v0.1.0_dryrun_20260811_051845.log
/oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update go

## 6. Create missing parent objects in production

This cell is blocked unless the configuration value is exactly:

```python
PARENT_REAL_CONFIRMATION = "CREATE_MISSING_ANALYSIS_PARENT_OBJECTS_IN_IGVF_PROD"
```

After authorization, each missing object is queried, dry-run, POSTed, queried
again, and compared with the intended metadata before the notebook proceeds to
the next dependency.
This version reuses an existing Portal `Software` object when the global `name` already exists under a different alias. It also reuses an existing `SoftwareVersion` matching the same software and version. A `409` on `software:name` should therefore not be retried as a new object.


In [13]:
PARENT_REAL_CONFIRMATION = "CREATE_MISSING_ANALYSIS_PARENT_OBJECTS_IN_IGVF_PROD"

parent_submission_results = submit_all_missing_parent_objects(PARENT_RECORDS)

parent_final_status = parent_status_table(PARENT_RECORDS)
parent_final_status_path = (
    PARENT_METADATA_DIR / "parent_object_status_after_submission.tsv"
)
parent_final_status.to_csv(
    parent_final_status_path,
    sep="\t",
    index=False,
    lineterminator="\n",
)

if not parent_final_status["verification"].eq("MATCH").all():
    raise RuntimeError(
        "Parent-object reconciliation is incomplete. Review "
        f"{parent_final_status_path}"
    )

print("\nAll required Software, SoftwareVersion, AnalysisStep, and "
      "AnalysisStepVersion records exist and match.")
print("Final status:", parent_final_status_path)

All parent objects already exist and match; no production POST is needed.

All required Software, SoftwareVersion, AnalysisStep, and AnalysisStepVersion records exist and match.
Final status: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/_upload_work/parent_metadata/parent_object_status_after_submission.tsv


## 7. Final read-only Portal preflight for the 16 file records

This verifies the four active AnalysisSets, reference file, and the two
resolved AnalysisStepVersion records. It also checks for pre-existing file
aliases and refuses to overwrite or PATCH mismatched records.

In [14]:
# Verify the exact parent chain required by the resolved file metadata.
core_identifiers = set(all_rows["file_set"].map(normalize_identifier))
core_identifiers.update(
    all_rows["analysis_step_version"].map(normalize_identifier)
)
for value in all_rows.get("reference_files", pd.Series(dtype=str)):
    core_identifiers.update(
        normalize_identifier(item)
        for item in split_identifiers(value)
    )
core_identifiers.discard("")

core_checks = []
for identifier in sorted(core_identifiers):
    obj = portal_get(identifier)
    core_checks.append({
        "identifier": identifier,
        "found": bool(obj),
        "accession": str(obj.get("accession") or "") if obj else "",
        "type": (
            ",".join(obj.get("@type", []))
            if obj and isinstance(obj.get("@type"), list)
            else str(obj.get("@type") or "") if obj else ""
        ),
        "status": str(obj.get("status") or "") if obj else "",
    })
    print(identifier, "FOUND" if obj else "NOT FOUND")

core_check_df = pd.DataFrame(core_checks)
core_check_path = UPLOAD_WORK_DIR / "core_dependency_preflight.tsv"
core_check_df.to_csv(
    core_check_path,
    sep="\t",
    index=False,
    lineterminator="\n",
)

if not core_check_df["found"].all():
    missing_ids = core_check_df.loc[
        ~core_check_df["found"],
        "identifier",
    ].tolist()
    raise RuntimeError(
        "Missing core Portal dependencies: "
        f"{missing_ids}. Complete the parent-object section above, then "
        f"rerun this cell. Report: {core_check_path}"
    )

# Confirm that the resolved AnalysisStepVersion aliases are the ones carried
# by the 16 file records.
observed_step_versions = set(
    all_rows["analysis_step_version"].map(normalize_identifier)
)
expected_step_versions = {
    normalize_identifier(PREPROCESS_ANALYSIS_STEP_VERSION_ALIAS),
    normalize_identifier(PERTURBNMF_ANALYSIS_STEP_VERSION_ALIAS),
}
if observed_step_versions != expected_step_versions:
    raise RuntimeError(
        "Resolved file metadata carries unexpected AnalysisStepVersion links: "
        f"observed={sorted(observed_step_versions)}, "
        f"expected={sorted(expected_step_versions)}"
    )

existing_rows = []
for row in all_rows.to_dict("records"):
    obj = portal_get(row["aliases"])
    issues = record_comparison(row, obj) if obj else []
    existing_rows.append({
        "day": row["_day"],
        "stage": row["_stage"],
        "profile": row["_profile"],
        "alias": row["aliases"],
        "exists": bool(obj),
        "accession": str(obj.get("accession") or "") if obj else "",
        "portal_url": (
            portal_url(str(obj.get("accession") or ""))
            if obj else ""
        ),
        "verification": (
            "MATCH" if obj and not issues
            else "MISMATCH" if obj
            else "NOT_FOUND"
        ),
        "issues": "; ".join(issues),
    })

existing_preflight = pd.DataFrame(existing_rows)
existing_preflight_path = (
    UPLOAD_WORK_DIR / "existing_alias_preflight.tsv"
)
existing_preflight.to_csv(
    existing_preflight_path,
    sep="\t",
    index=False,
    lineterminator="\n",
)
if existing_preflight["verification"].eq("MISMATCH").any():
    raise RuntimeError(
        "At least one file alias already exists with mismatched metadata. "
        f"Review {existing_preflight_path}; do not POST or PATCH automatically."
    )

print("\nExisting-file-alias preflight:")
print(
    existing_preflight.groupby(
        ["stage", "verification"]
    ).size().to_string()
)
print("Report:", existing_preflight_path)

IGVFDS0463XTUH FOUND
IGVFDS0991DBQZ FOUND
IGVFDS2307REVJ FOUND
IGVFDS4402ZGWJ FOUND
IGVFFI7217ZMJZ FOUND
jesse-engreitz:analysis_step_version_WTC11-EC_300gene_CC-Perturbseq_preprocessing_v0.1.0 FOUND
jesse-engreitz:analysis_step_version_WTC11-EC_300gene_PerturbNMF_gene_programs_v0.1.0 FOUND

Existing-file-alias preflight:
stage    verification
h5ad     NOT_FOUND       4
h5mu     NOT_FOUND       4
tabular  NOT_FOUND       8
Report: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/_upload_work/existing_alias_preflight.tsv


## 8. Dry-run the four h5ad MatrixFiles

Every row is queried first. Existing matching records are reused; missing
records are dry-run individually.

In [15]:
h5ad_dryrun_results = dry_run_rows(h5ad, "01_h5ad")
if not h5ad_dryrun_results["status"].isin({
    "DRY_RUN_PASS", "ALREADY_EXISTS_MATCH"
}).all():
    raise RuntimeError("The h5ad dry-run stage did not pass completely")
print("\nH5AD dry-run stage passed")

Command: /oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/bin/python /oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/igvf_utils/igvf_utils/MetaDataRegistration/iu_register.py -m prod -p matrix_file -d -i /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/_upload_work/one_row_tsv/01_matrix_file_jesse-engreitz_matrix_file_WTC11-EC_300gene_CC-Perturbseq_D0_h5ad.tsv
Log: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/logs/igvf_utils_analysis_upload/01_matrix_file_jesse-engreitz_matrix_file_WTC11-EC_300gene_CC-Perturbseq_D0_h5ad_dryrun_20260811_052713.log
/oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then up

## 9. Real submission of the four h5ad MatrixFiles

This cell remains blocked until:

```python
H5AD_REAL_CONFIRMATION = "UPLOAD_4_H5AD_FILES_TO_IGVF_PROD"
```

The notebook verifies each Portal record after submission and refuses blind
retries after a partial client failure.

In [16]:
H5AD_REAL_CONFIRMATION = "UPLOAD_4_H5AD_FILES_TO_IGVF_PROD"

if H5AD_REAL_CONFIRMATION != EXPECTED_H5AD_TOKEN:
    raise RuntimeError(
        "Real h5ad submission is blocked. Review the dry runs, then set "
        f"H5AD_REAL_CONFIRMATION exactly to {EXPECTED_H5AD_TOKEN!r}."
    )

h5ad_submit_results = submit_rows(h5ad, "01_h5ad")
print("\nH5AD real-submission stage completed")

Command: /oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/bin/python /oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/igvf_utils/igvf_utils/MetaDataRegistration/iu_register.py -m prod -p matrix_file -i /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/_upload_work/one_row_tsv/01_matrix_file_jesse-engreitz_matrix_file_WTC11-EC_300gene_CC-Perturbseq_D0_h5ad.tsv
Log: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/logs/igvf_utils_analysis_upload/01_matrix_file_jesse-engreitz_matrix_file_WTC11-EC_300gene_CC-Perturbseq_D0_h5ad_submit_20260811_052721.log
/oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then updat

## 10. Verify all four h5ad records before dependent dry runs

In [17]:
h5ad_reconciliation = reconcile_rows(h5ad, "h5ad_portal_reconciliation.csv")
if len(h5ad_reconciliation) != 4:
    raise AssertionError("Expected four h5ad reconciliation rows")
if not h5ad_reconciliation["verification"].eq("MATCH").all():
    raise RuntimeError(
        "Not all h5ad records are present and matching. Do not proceed to dependent files."
    )
if h5ad_reconciliation["accession"].nunique() != 4:
    raise RuntimeError("The four h5ad aliases did not resolve to four unique accessions")
print("All four h5ad dependencies are present and matched")

Reconciliation: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/metadata/h5ad_portal_reconciliation.csv
day     profile                                                             alias      accession verification portal_upload_status portal_object_status
    matrix_file jesse-engreitz:matrix_file_WTC11-EC_300gene_CC-Perturbseq_D0_h5ad IGVFFI7756RKHT        MATCH              pending          in progress
    matrix_file jesse-engreitz:matrix_file_WTC11-EC_300gene_CC-Perturbseq_D1_h5ad IGVFFI4783YGLZ        MATCH              pending          in progress
    matrix_file jesse-engreitz:matrix_file_WTC11-EC_300gene_CC-Perturbseq_D2_h5ad IGVFFI5165BSGC        MATCH              pending          in progress
    matrix_file jesse-engreitz:matrix_file_WTC11-EC_300gene_CC-Perturbseq_D3_h5ad IGVFFI9086SWZC        MATCH              pending          in progress
All four h5ad dependencies are present and matched


## 11. Dry-run the four h5mu and eight tabular files

The notebook first confirms that every immediate `derived_from` h5ad record
exists in production.

In [18]:
dependent = pd.concat([h5mu, tabular], ignore_index=True)

for row in dependent.to_dict("records"):
    for identifier in split_identifiers(row.get("derived_from", "")):
        obj = portal_get(identifier)
        if not obj:
            raise RuntimeError(
                f"Missing derived_from dependency {identifier} for {row['aliases']}"
            )

# Preserve the intended order: h5mu first, then guide quantifications, then QC.
dependent_dryrun_results = dry_run_rows(dependent, "02_dependent")
if not dependent_dryrun_results["status"].isin({
    "DRY_RUN_PASS", "ALREADY_EXISTS_MATCH"
}).all():
    raise RuntimeError("The dependent-file dry-run stage did not pass completely")
print("\nDependent-file dry-run stage passed")

Command: /oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/bin/python /oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/igvf_utils/igvf_utils/MetaDataRegistration/iu_register.py -m prod -p matrix_file -d -i /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/_upload_work/one_row_tsv/01_matrix_file_jesse-engreitz_matrix_file_WTC11-EC_300gene_PerturbNMF_D0_h5mu.tsv
Log: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/logs/igvf_utils_analysis_upload/01_matrix_file_jesse-engreitz_matrix_file_WTC11-EC_300gene_PerturbNMF_D0_h5mu_dryrun_20260811_052857.log
/oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update g

## 12. Real submission of the 12 dependent files

This cell remains blocked until:

```python
DEPENDENT_REAL_CONFIRMATION = "UPLOAD_12_DEPENDENT_FILES_TO_IGVF_PROD"
```

In [20]:
DEPENDENT_REAL_CONFIRMATION = "UPLOAD_12_DEPENDENT_FILES_TO_IGVF_PROD"

if DEPENDENT_REAL_CONFIRMATION != EXPECTED_DEPENDENT_TOKEN:
    raise RuntimeError(
        "Real dependent-file submission is blocked. Review the dry runs, then set "
        f"DEPENDENT_REAL_CONFIRMATION exactly to {EXPECTED_DEPENDENT_TOKEN!r}."
    )

dependent_submit_results = submit_rows(dependent, "02_dependent")
print("\nDependent-file real-submission stage completed")

Command: /oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/bin/python /oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/igvf_utils/igvf_utils/MetaDataRegistration/iu_register.py -m prod -p matrix_file -i /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/_upload_work/one_row_tsv/01_matrix_file_jesse-engreitz_matrix_file_WTC11-EC_300gene_PerturbNMF_D0_h5mu.tsv
Log: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/logs/igvf_utils_analysis_upload/01_matrix_file_jesse-engreitz_matrix_file_WTC11-EC_300gene_PerturbNMF_D0_h5mu_submit_20260811_052946.log
/oak/stanford/groups/engreitz/Users/tri/envs/igvf_utils/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update goog

## 13. Final Portal reconciliation and accession export

The final CSV checks all 16 aliases against the expected AnalysisSet, MD5,
content type, file format, and resolved AnalysisStepVersion.

In [21]:
final_reconciliation = reconcile_rows(
    all_rows,
    "igvf_analysis_file_accessions_16.csv",
)

found = int(final_reconciliation["accession"].astype(bool).sum())
matched = int(final_reconciliation["verification"].eq("MATCH").sum())
unique_accessions = int(
    final_reconciliation.loc[
        final_reconciliation["accession"].astype(bool), "accession"
    ].nunique()
)

summary_lines = [
    "CC-Perturb-seq IGVF analysis-file submission reconciliation",
    f"checked_utc: {now_utc()}",
    f"preprocessing_version: {prep['portal_version']}",
    f"preprocessing_commit_sha: {prep['commit_sha']}",
    f"preprocessing_source_url: {prep['software_version_source_url']}",
    f"perturbnmf_version: {pnmf['portal_version']}",
    f"perturbnmf_commit_sha: {pnmf['commit_sha']}",
    f"perturbnmf_source_url: {pnmf['software_version_source_url']}",
    f"preprocessing_analysis_step_version: {PREPROCESS_ANALYSIS_STEP_VERSION_ALIAS}",
    f"perturbnmf_analysis_step_version: {PERTURBNMF_ANALYSIS_STEP_VERSION_ALIAS}",
    "expected_records: 16",
    f"found_records: {found}",
    f"metadata_matches: {matched}",
    f"unique_accessions: {unique_accessions}",
    f"accession_csv: {METADATA_DIR / 'igvf_analysis_file_accessions_16.csv'}",
    "",
    "Portal object-status counts:",
]
for key, value in final_reconciliation["portal_object_status"].value_counts(dropna=False).items():
    summary_lines.append(f"  {key or '<blank>'}: {value}")
summary_lines.append("")
summary_lines.append("Portal upload-status counts:")
for key, value in final_reconciliation["portal_upload_status"].value_counts(dropna=False).items():
    summary_lines.append(f"  {key or '<blank>'}: {value}")

summary_path = METADATA_DIR / "igvf_analysis_file_upload_summary.txt"
summary_path.write_text("\n".join(summary_lines) + "\n", encoding="utf-8")
print("\n".join(summary_lines))

if found == 16 and matched == 16 and unique_accessions == 16:
    completion_marker = OUTPUT_ROOT / "IGVF_UPLOAD_COMPLETE_16_FILES.txt"
    completion_marker.write_text(
        "\n".join([
            "CC-Perturb-seq IGVF analysis-file records: COMPLETE AND MATCHED",
            f"checked_utc: {now_utc()}",
            f"preprocessing_version: {prep['portal_version']}",
            f"preprocessing_commit_sha: {prep['commit_sha']}",
            f"perturbnmf_version: {pnmf['portal_version']}",
            f"perturbnmf_commit_sha: {pnmf['commit_sha']}",
            "records_found: 16",
            "metadata_matches: 16",
            "unique_accessions: 16",
            f"accession_csv: {METADATA_DIR / 'igvf_analysis_file_accessions_16.csv'}",
            "Note: downstream DACC/checkfiles validation may still be pending.",
            "",
        ]),
        encoding="utf-8",
    )
    print("\nCompletion marker:", completion_marker)
else:
    raise RuntimeError(
        "Final reconciliation is incomplete. Review "
        f"{METADATA_DIR / 'igvf_analysis_file_accessions_16.csv'}"
    )

Reconciliation: /oak/stanford/groups/engreitz/Projects/IGVF/CC-Perturb-seq/300gene/analysis_submission/metadata/igvf_analysis_file_accessions_16.csv
day      profile                                                                   alias      accession verification portal_upload_status portal_object_status
 D0  matrix_file       jesse-engreitz:matrix_file_WTC11-EC_300gene_CC-Perturbseq_D0_h5ad IGVFFI7756RKHT        MATCH              pending          in progress
 D1  matrix_file       jesse-engreitz:matrix_file_WTC11-EC_300gene_CC-Perturbseq_D1_h5ad IGVFFI4783YGLZ        MATCH              pending          in progress
 D2  matrix_file       jesse-engreitz:matrix_file_WTC11-EC_300gene_CC-Perturbseq_D2_h5ad IGVFFI5165BSGC        MATCH              pending          in progress
 D3  matrix_file       jesse-engreitz:matrix_file_WTC11-EC_300gene_CC-Perturbseq_D3_h5ad IGVFFI9086SWZC        MATCH              pending          in progress
 D0  matrix_file          jesse-engreitz:matrix_file_WTC

## Principal outputs

Hard-coded first-pass software and parent provenance:

```text
${OUTPUT_ROOT}/_upload_work/software_version_resolution.tsv
${OUTPUT_ROOT}/_upload_work/resolved_software_provenance.json
${OUTPUT_ROOT}/_upload_work/parent_metadata/01_software.tsv
${OUTPUT_ROOT}/_upload_work/parent_metadata/02_software_version.tsv
${OUTPUT_ROOT}/_upload_work/parent_metadata/03_analysis_step.tsv
${OUTPUT_ROOT}/_upload_work/parent_metadata/04_analysis_step_version.tsv
${OUTPUT_ROOT}/_upload_work/parent_metadata/parent_object_status.tsv
${OUTPUT_ROOT}/_upload_work/parent_metadata/parent_object_status_after_submission.tsv
```

The version report will show:

```text
preprocessing  v0.1.0  hardcoded_first_pass
perturbnmf     v0.1.0  hardcoded_first_pass
```

No Git clone, fetch, GitHub API call, exact tag lookup, or commit lookup occurs.
The repository root URLs are recorded. Before final public release, reconcile
these SoftwareVersion records with the exact tag or commit if DACC requires it.

Resolved file metadata:

```text
${OUTPUT_ROOT}/_upload_work/resolved_file_metadata/
    upload_manifest_16_files_resolved_versions.tsv
    01_matrix_file_h5ad_resolved_versions.tsv
    02_matrix_file_h5mu_resolved_versions.tsv
    03_tabular_file_step1_resolved_versions.tsv
```

Upload logs:

```text
${OUTPUT_ROOT}/logs/igvf_utils_analysis_upload/
```

Final accession table:

```text
${OUTPUT_ROOT}/metadata/igvf_analysis_file_accessions_16.csv
```

Final completion marker:

```text
${OUTPUT_ROOT}/IGVF_UPLOAD_COMPLETE_16_FILES.txt
```

The notebook does not PATCH existing records and refuses to overwrite a
mismatched Portal object. If a hard-coded parent alias already exists but its
metadata differs, the reconciliation cell will stop for review.
